In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

mu, sigma = 0, 1
n_samples = 1000
np.random.seed(42)
samples = np.random.normal(mu, sigma, n_samples)

running_means = np.cumsum(samples) / np.arange(1, n_samples + 1)

#welford's algo O(n²) -> O(n)
M = np.zeros(n_samples)
S = np.zeros(n_samples)

M[0] = samples[0]
for i in range(1, n_samples):
    delta = samples[i] - M[i-1]
    M[i] = M[i-1] + delta/ (i + 1)
    S[i] = S[i-1] + delta* (samples[i] + M[i])
running_vars = np.where(np.arange(n_samples < 1), np.nan, S / np.arange(n_samples))
running_vars = [np.var(samples[:i], ddof=1) for i in range(1, n_samples + 1)]

frames = []
for i in range(10, n_samples):
    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=np.arange(1, i + 1),
                y=running_means[:i],
                mode='lines',
                line=dict(color='#39ff14', width=3),
                name='Sample Mean'
            ),
            go.Scatter(
                x=np.arange(1, i + 1),
                y=[mu] * i,
                mode='lines',
                line=dict(color='yellow', width=2, dash='dot'),
                name='Theoretical μ'
            ),
            go.Scatter(
                x=np.arange(1, i + 1),
                y=running_vars[:i],
                mode='lines',
                line=dict(color='#00ffff', width=3),
                name='Sample Var'
            ),
            go.Scatter(
                x=np.arange(1, i + 1),
                y=[sigma**2] * i,
                mode='lines',
                line=dict(color='magenta', width=2, dash='dot'),
                name='Theoretical σ²'
            ),
        ],
        layout=go.Layout(
            xaxis=dict(range=[0, i + 20]),
            xaxis2=dict(range=[0, i + 20])
        ),
        name=f'frame{i}'
    ))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Convergence of Sample Mean', 'Convergence of Sample Variance'),
    column_widths=[0.5, 0.5]
)

fig.add_trace(
    go.Scatter(
        x=[1],
        y=[running_means[0]],
        mode='lines',
        line=dict(color='#39ff14', width=3),
        name='Sample Mean'
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=[1],
        y=[mu],
        mode='lines',
        line=dict(color='yellow', width=2, dash='dot'),
        name='Theoretical μ'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=[1],
        y=[running_vars[0]],
        mode='lines',
        line=dict(color='#00ffff', width=3),
        name='Sample Var'
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=[1],
        y=[sigma**2],
        mode='lines',
        line=dict(color='magenta', width=2, dash='dot'),
        name='Theoretical σ²'
    ),
    row=1, col=2
)

fig.frames = frames

fig.update_layout(
    height=500,
    width=1000,
    title_text="Law of Large Numbers: Convergence of Sample Mean and Variance",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=False,
    updatemenus=[{
        'type': 'buttons',
        'x': 0.5, 'y': -0.1,
        'showactive': False,
        'xanchor': 'center',
        'yanchor': 'top',
        'buttons': [{
            'label': '▶ Play',
            'method': 'animate',
            'args': [None, {
                'frame': {'duration': 20, 'redraw': True},
                'fromcurrent': True,
                'transition': {'duration': 0}
            }]
        }]
    }]
)

for c in [1, 2]:
    fig.update_xaxes(
        title_text='Sample Size (n)',
        showgrid=True, gridcolor='rgba(128,128,128,0.3)', row=1, col=c
    )
    fig.update_yaxes(
        showgrid=True, gridcolor='rgba(128,128,128,0.3)', row=1, col=c
    )

fig.update_yaxes(title_text='Sample Mean', row=1, col=1)
fig.update_yaxes(title_text='Sample Variance', row=1, col=2)

fig.show()
